# 02 - Chuẩn bị dữ liệu

**Phase 1 | OCR bán giám sát cho chữ viết tay tiếng Việt**

## Mục tiêu

Notebook này thực hiện pipeline chuẩn bị dữ liệu cho Text Line Dataset.


## Điều chỉnh: Đồng bộ split dữ liệu với VietOCR Trainer

**Thay đổi:** Loại bỏ split 3 nhánh (train/val/test) và chuyển sang split 2 nhánh (90% train / 10% validation).

**Lý do:**
- `Trainer` của VietOCR chỉ nhận hai đường dẫn annotation: `train_annotation` và `valid_annotation`, không có `test_annotation` trong config.
- Split 80/10/10 trước đây tạo `test_line.txt` nhưng VietOCR không dùng trực tiếp, gây lệch nghĩa giữa bước chuẩn bị dữ liệu và bước huấn luyện.
- Loại bỏ split dư giúp pipeline đơn giản và nhất quán hơn.

**Chiến lược đánh giá:** Đánh giá hold-out nên thực hiện sau huấn luyện (ví dụ trong notebook `04_evaluate_baseline.ipynb`) trên tập validation/đánh giá chuyên biệt hoặc qua cross-validation, thay vì đóng cứng ở bước chuẩn bị dữ liệu.

## Điều chỉnh: Loại bỏ xử lý từ vựng thủ công

**Thay đổi:** Loại bỏ `build_vocab()`, bước sinh `vocab.txt`, và toàn bộ kiểm tra liên quan vocabulary thủ công.

**Lý do:**
- `Cfg.load_config_from_name("vgg_seq2seq")` của VietOCR đã có sẵn bộ ký tự tiếng Việt đầy đủ, tương thích pretrained weights.
- Tạo vocabulary tùy chỉnh từ train data (168 ký tự) rồi ghi đè `config["vocab"]` dễ gây lệch kích thước embedding decoder do khác `num_classes`.
- Loại bỏ bước này giúp tránh lỗi shape mismatch ngay từ data preparation.

**Ý nghĩa:**
- Data preparation chỉ tạo `train_line.txt`, `val_line.txt`, `split_metadata.json`.
- Không cần sinh hay dùng `vocab.txt`.
- Vocabulary được VietOCR quản lý nội bộ ở bước training qua `Cfg.load_config_from_name()`.

In [1]:
# ============================================================
# CẤU HÌNH
# ============================================================

CONFIG = {
    # Input đường dẫn (Text Line only — Word Dataset dropped for Phase 1)
    "textline_labels": "../Dataset/labels.txt",
    "textline_data_dir": "../Dataset/data/",

    # Output đường dẫn
    "output_dir": "../data/processed/",
    "train_line_path": "../data/processed/train_line.txt",
    "val_line_path": "../data/processed/val_line.txt",
    "test_line_path": "../data/processed/test_line.txt",
    "metadata_path": "../data/processed/split_metadata.json",

    # Split parameters (train/val/test = 80/10/10)
    "train_ratio": 0.8,
    "val_ratio": 0.1,
    "test_ratio": 0.1,
    "random_seed": 42,

    # xác thực
    "min_image_dim": 16,
    "aspect_ratio_min": 0.5,   # loại ảnh có W/H < 0.5
    "aspect_ratio_max": 50.0,  # loại ảnh có W/H > 50.0

    # label cleaning
    "min_label_len": 2,  # loại nhãn có len(text.strip()) < 2
}

assert abs(CONFIG["train_ratio"] + CONFIG["val_ratio"] + CONFIG["test_ratio"] - 1.0) < 1e-9, \
    "Split ratios must sum to 1.0"

print("Đã nạp CONFIG.")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

Đã nạp CONFIG.
  textline_labels: ../Dataset/labels.txt
  textline_data_dir: ../Dataset/data/
  output_dir: ../data/processed/
  train_line_path: ../data/processed/train_line.txt
  val_line_path: ../data/processed/val_line.txt
  test_line_path: ../data/processed/test_line.txt
  metadata_path: ../data/processed/split_metadata.json
  train_ratio: 0.8
  val_ratio: 0.1
  test_ratio: 0.1
  random_seed: 42
  min_image_dim: 16
  aspect_ratio_min: 0.5
  aspect_ratio_max: 50.0
  min_label_len: 2


In [2]:
# ============================================================
# Imports
# ============================================================

import os
import re
import json
import unicodedata
from datetime import datetime
from pathlib import Path
from collections import Counter

from PIL import Image
from sklearn.model_selection import train_test_split

print("Đã nạp tất cả Imports thành công.")

Đã nạp tất cả Imports thành công.


In [4]:
# ============================================================
# Helper Functions
# ============================================================


def nfc_normalize(text: str) -> str:
    """Apply Unicode NFC normalization to text.

    Critical for Vietnamese which has 134 accented characters.
    NFC composes characters (e.g., base + combining accent -> single codepoint),
    preventing silent vocabulary misses when the same visual character has
    different binary representations (NFC vs NFD).

    Args:
        text: Input string, possibly in NFD or mixed normalization form.

    Returns:
        NFC-normalized string.
    """
    return unicodedata.normalize("NFC", text)


def clean_transcription(text: str) -> str:
    """Clean a transcription string.

    Steps:
    1. Remove Unicode control characters (category Cc) and format
       characters (category Cf) — these are non-printable and should
       never appear in valid ground-truth labels.
    2. Collapse runs of whitespace (spaces, tabs, etc.) into a single
       space character.
    3. Strip leading and trailing whitespace.

    Args:
        text: Raw transcription string (should already be NFC-normalized).

    Returns:
        Cleaned transcription string.
    """
    # Step 1: Remove control (Cc) and format (Cf) characters
    cleaned = "".join(
        ch for ch in text
        if unicodedata.category(ch) not in ("Cc", "Cf")
    )
    # Step 2: Collapse multiple whitespace into single space
    cleaned = re.sub(r"\s+", " ", cleaned)
    # Step 3: Strip leading/trailing whitespace
    cleaned = cleaned.strip()
    return cleaned


def parse_label_file(filepath: str) -> list:
    """Read a tab-separated label file and return (image_path, transcription) tuples.

    Expected format per line:  data/filename.ext\ttranscription

    Handles edge cases:
    - Empty lines are skipped.
    - Lines without a tab character are skipped with a warning.

    Args:
        filepath: Path to the labels.txt file.

    Returns:
        List of (image_path, transcription) tuples.
    """
    entries = []
    skipped = 0
    with open(filepath, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.rstrip("\n").rstrip("\r")
            if not line.strip():
                skipped += 1
                continue
            if "\t" not in line:
                print(f"  [WARN] Line {line_num}: no tab found, skipping: {line[:80]}")
                skipped += 1
                continue
            parts = line.split("\t", maxsplit=1)
            image_path = parts[0].strip()
            transcription = parts[1] if len(parts) > 1 else ""
            entries.append((image_path, transcription))
    if skipped > 0:
        print(f"  [INFO] Skipped {skipped} lines in {filepath}")
    return entries


def validate_images(entries: list, data_dir: str) -> tuple:
    """Validate image files: existence, integrity, minimum dimensions, and aspect ratio.

    For each (image_path, label) entry:
    1. Check that the file exists on disk (os.path.exists).
    2. Open with PIL and call .verify() for integrity.
    3. Re-open to read dimensions, check both >= min_image_dim.
    4. Check aspect ratio (W/H) is within [aspect_ratio_min, aspect_ratio_max].

    Args:
        entries: List of (image_path, transcription) tuples. image_path is
                 relative (e.g., 'data/file.png').
        data_dir: Base directory. The actual file path is constructed as
                  data_dir + filename (the part after 'data/' in image_path).

    Returns:
        (valid_entries, invalid_entries) where each is a list of
        (image_path, transcription, error_reason_or_None) tuples.
    """
    valid = []
    invalid = []
    min_dim = CONFIG["min_image_dim"]
    ar_min = CONFIG["aspect_ratio_min"]
    ar_max = CONFIG["aspect_ratio_max"]

    for idx, (img_path, label) in enumerate(entries):
        # Construct the full đường dẫn on disk
        # image_path format: 'data/filename.ext' -> strip 'data/' prefix
        if img_path.startswith("data/"):
            filename = img_path[len("data/"):]
        else:
            filename = img_path
        full_path = os.path.join(data_dir, filename)

        # Kiểm tra 1: File existence
        if not os.path.exists(full_path):
            invalid.append((img_path, label, "file_not_found"))
            continue

        # Kiểm tra 2: tính toàn vẹn ảnh via PIL.verify()
        try:
            with Image.open(full_path) as img:
                img.verify()
        except Exception as e:
            invalid.append((img_path, label, f"verify_failed: {e}"))
            continue

        # Kiểm tra 3: Dimensions (need to re-open after verify)
        try:
            with Image.open(full_path) as img:
                w, h = img.size
                if w < min_dim or h < min_dim:
                    invalid.append((img_path, label, f"too_small: {w}x{h}"))
                    continue
                # Kiểm tra 4: Aspect ratio
                aspect_ratio = w / h
                if aspect_ratio < ar_min or aspect_ratio > ar_max:
                    invalid.append((img_path, label,
                                    f"aspect_ratio: {aspect_ratio:.2f} (W={w}, H={h})"))
                    continue
        except Exception as e:
            invalid.append((img_path, label, f"open_failed: {e}"))
            continue

        valid.append((img_path, label))

        # Progress indicator
        if (idx + 1) % 5000 == 0:
            print(f"  Validated {idx + 1}/{len(entries)} ...")

    return valid, invalid


def extract_source(image_path: str) -> str:
    """Extract the source prefix from an image filename.

    Convention: filenames are formatted as  SOURCE__rest_of_name.ext
    e.g.  'data/VNOnDB_line__0001.png'  ->  'VNOnDB_line'
          'data/UIT_HWDB_line_flat__0042.jpg'  ->  'UIT_HWDB_line_flat'

    Args:
        image_path: The image path string (e.g., 'data/VNOnDB_line__xxx.png').

    Returns:
        Source prefix string.
    """
    # Get the filename part (after last /)
    basename = os.path.basename(image_path)
    # chia on double underscore
    if "__" in basename:
        return basename.split("__")[0]
    else:
        return "unknown"


print("Helper functions defined:")
print("  - nfc_normalize(text)")
print("  - clean_transcription(text)")
print("  - parse_label_file(filepath)")
print("  - validate_images(entries, data_dir)")
print("  - extract_source(image_path)")

Helper functions defined:
  - nfc_normalize(text)
  - clean_transcription(text)
  - parse_label_file(filepath)
  - validate_images(entries, data_dir)
  - extract_source(image_path)


## 1. Parse file nhãn


In [7]:
# --------------------------------------------------
# 1. Parse Label File (Text Line only)
# --------------------------------------------------

print("Đang parse nhãn Text Line...")
textline_entries = parse_label_file(CONFIG["textline_labels"])
print(f"  Text Line entries: {len(textline_entries):,}")

# Hiển thị first 5 entries
print("\n--- First 5 Text Line entries ---")
for img, txt in textline_entries[:5]:
    print(f"  {img}  ->  {txt[:80]}{'...' if len(txt) > 80 else ''}")

Đang parse nhãn Text Line...
  Text Line entries: 16,363

--- First 5 Text Line entries ---
  data/vn_handwritten_images__1.jpg  ->  Số 3 Nguyễn Ngọc Vũ, Hà Nội
  data/vn_handwritten_images__2.jpg  ->  Số 30 Nguyên Hồng, Láng Hạ, Đống Đa, Hà Nội
  data/vn_handwritten_images__3.jpg  ->  58 Thái Thịnh, Đống Đa, Hà Nội
  data/vn_handwritten_images__4.jpeg  ->  Số 370/8 khu phố 5B, phường Tân Biên, Biên Hòa, Đồng Nai
  data/vn_handwritten_images__5.jpg  ->  Vĩnh Trung Plaza, B, 255-257 đường Hùng Vương, phường Vĩnh Trung


## 2. Chuẩn hóa Unicode NFC


In [8]:
# --------------------------------------------------
# 2. Unicode NFC Normalization (Text Line only)
# --------------------------------------------------

def apply_nfc_and_count(entries: list) -> tuple:
    """Apply NFC normalization to all transcriptions, count changes."""
    normalized = []
    changed_count = 0
    changed_examples = []
    for img_path, text in entries:
        nfc_text = nfc_normalize(text)
        if nfc_text != text:
            changed_count += 1
            if len(changed_examples) < 10:  # Collect up to 10 examples
                changed_examples.append((img_path, text, nfc_text))
        normalized.append((img_path, nfc_text))
    return normalized, changed_count, changed_examples


# Apply to Text Line
print("Đang áp dụng NFC normalization cho transcription Text Line...")
textline_entries, tl_changed, tl_examples = apply_nfc_and_count(textline_entries)
print(f"  Text Line: {tl_changed:,} entries changed out of {len(textline_entries):,}")

# Show before/after comparison for up to 10 entries
if tl_examples:
    print(f"\n--- Before/After NFC comparison ({min(len(tl_examples), 10)} entries) ---")
    for img, before, after in tl_examples[:10]:
        print(f"  File: {img}")
        print(f"    BEFORE: {before[:60]}  (len={len(before)})")
        print(f"    AFTER:  {after[:60]}  (len={len(after)})")
        print(f"    Codepoints changed: {len(before)} -> {len(after)}")
        print()
else:
    print("\n  Không có transcription nào thay đổi sau NFC normalization.")
    print("  (Tất cả transcription đã ở dạng NFC.)")

# Assert all transcriptions are now NFC
for img, text in textline_entries:
    assert text == unicodedata.normalize("NFC", text), f"NFC fail: {img}"

print("Assertion passed: TẤT CẢ transcription đã NFC-normalized.")

Đang áp dụng NFC normalization cho transcription Text Line...
  Text Line: 0 entries changed out of 16,363

  Không có transcription nào thay đổi sau NFC normalization.
  (Tất cả transcription đã ở dạng NFC.)
Assertion passed: TẤT CẢ transcription đã NFC-normalized.


## 2b. Làm sạch nhãn (whitespace, control chars, ràng buộc độ dài)

Theo khuyến nghị từ EDA (xem `docs/phase1_notes.md`):
- Loại ký tự điều khiển/không in được (Unicode category `Cc`, `Cf`).
- Chuẩn hóa khoảng trắng: co cụm nhiều khoảng trắng về 1 khoảng trắng và `strip()` hai đầu.
- Loại mẫu có nhãn quá ngắn: `len(text.strip()) < 2`.

In [9]:
# --------------------------------------------------
# 2b. Label Cleaning: whitespace, control chars, length guard
# --------------------------------------------------

min_label_len = CONFIG["min_label_len"]
before_count = len(textline_entries)

# Apply clean_transcription and track changes
cleaned_entries = []
ws_changed = 0       # entries where whitespace was normalized
ctrl_removed = 0     # entries where control chars were removed
too_short = 0        # entries dropped due to len < min_label_len
cleaning_examples = []

for img_path, text in textline_entries:
    # Detect control characters before cleaning
    has_ctrl = any(unicodedata.category(ch) in ("Cc", "Cf") for ch in text)
    if has_ctrl:
        ctrl_removed += 1

    cleaned = clean_transcription(text)

    # Detect whitespace normalization (ignoring control char removal)
    text_no_ctrl = "".join(ch for ch in text if unicodedata.category(ch) not in ("Cc", "Cf"))
    if cleaned != text_no_ctrl.strip():
        ws_changed += 1

    # Length guard
    if len(cleaned) < min_label_len:
        too_short += 1
        if len(cleaning_examples) < 5:
            cleaning_examples.append((img_path, text, cleaned, "too_short"))
        continue

    # Collect examples of actual changes
    if cleaned != text and len(cleaning_examples) < 10:
        cleaning_examples.append((img_path, text, cleaned, "cleaned"))

    cleaned_entries.append((img_path, cleaned))

textline_entries = cleaned_entries

print(f"Label Cleaning hoàn tất:")
print(f"  Entries trước: {before_count:,}")
print(f"  Entries sau:   {len(textline_entries):,}")
print(f"  Whitespace normalized: {ws_changed:,}")
print(f"  Control chars removed: {ctrl_removed:,}")
print(f"  Dropped (len < {min_label_len}): {too_short:,}")

if cleaning_examples:
    print(f"\n--- Ví dụ thay đổi (tối đa 10) ---")
    for img, before, after, reason in cleaning_examples:
        print(f"  [{reason}] {img}")
        print(f"    BEFORE: '{before[:60]}' (len={len(before)})")
        print(f"    AFTER:  '{after[:60]}' (len={len(after)})")
else:
    print("\n  Không có thay đổi nào — dữ liệu đã sạch.")

Label Cleaning hoàn tất:
  Entries trước: 16,363
  Entries sau:   16,363
  Whitespace normalized: 0
  Control chars removed: 0
  Dropped (len < 2): 0

  Không có thay đổi nào — dữ liệu đã sạch.


## 3. Kiểm tra tính hợp lệ ảnh


In [10]:
# --------------------------------------------------
# 3. Image xác thực (Text Line only)
# --------------------------------------------------

print("Đang validate ảnh Text Line...")
textline_valid, textline_invalid = validate_images(
    textline_entries, CONFIG["textline_data_dir"]
)
print(f"  Valid:   {len(textline_valid):,}")
print(f"  Invalid: {len(textline_invalid):,}")

# Hiển thị invalid entries (if any)
if textline_invalid:
    print(f"\n--- Invalid Entries ({len(textline_invalid)}) ---")
    for img, label, reason in textline_invalid[:20]:  # Show up to 20
        print(f"  {img} | reason: {reason} | label: {label[:40]}")
    if len(textline_invalid) > 20:
        print(f"  ... and {len(textline_invalid) - 20} more.")
else:
    print("\n  Tất cả ảnh đều hợp lệ!")

# Use only valid entries from here on
textline_entries = textline_valid

print(f"\nTiếp tục với:")
print(f"  Text Line: {len(textline_entries):,} valid entries")

Đang validate ảnh Text Line...
  Validated 5000/16363 ...
  Validated 10000/16363 ...
  Validated 15000/16363 ...
  Valid:   16,363
  Invalid: 0

  Tất cả ảnh đều hợp lệ!

Tiếp tục với:
  Text Line: 16,363 valid entries


## 4. Chia tập train / validation theo stratified (Text Line)

In [11]:
# --------------------------------------------------
# 4. Stratified 80/10/10 split (Text Line only)
# --------------------------------------------------

# Extract source prefix for each text-line entry
sources = [extract_source(img_path) for img_path, _ in textline_entries]

# Show source distribution
source_counts = Counter(sources)
print("Phân phối nguồn dữ liệu (Text Line):")
for src, cnt in sorted(source_counts.items()):
    print(f"  {src}: {cnt:,}")
print(f"  Total: {len(textline_entries):,}")

# Stage 1: tách train (80%) và holdout (20%)
holdout_ratio = CONFIG["val_ratio"] + CONFIG["test_ratio"]
train_line, holdout_line, train_sources, holdout_sources = train_test_split(
    textline_entries,
    sources,
    test_size=holdout_ratio,
    random_state=CONFIG["random_seed"],
    stratify=sources,
)

# Stage 2: tách holdout thành val/test theo tỷ lệ 10/10
val_ratio_within_holdout = CONFIG["val_ratio"] / holdout_ratio
val_line, test_line = train_test_split(
    holdout_line,
    test_size=(1 - val_ratio_within_holdout),
    random_state=CONFIG["random_seed"],
    stratify=holdout_sources,
)

print(f"\nKết quả split:")
print(f"  Train: {len(train_line):,}  ({len(train_line)/len(textline_entries)*100:.1f}%)")
print(f"  Val:   {len(val_line):,}  ({len(val_line)/len(textline_entries)*100:.1f}%)")
print(f"  Test:  {len(test_line):,}  ({len(test_line)/len(textline_entries)*100:.1f}%)")

# Detailed per-source breakdown
train_sources = Counter(extract_source(img) for img, _ in train_line)
val_sources = Counter(extract_source(img) for img, _ in val_line)
test_sources = Counter(extract_source(img) for img, _ in test_line)

print(f"\n{'Source':<30} {'Total':>7} {'Train':>7} {'Val':>7} {'Test':>7}")
print("-" * 64)
for src in sorted(source_counts.keys()):
    total = source_counts[src]
    tr = train_sources.get(src, 0)
    va = val_sources.get(src, 0)
    te = test_sources.get(src, 0)
    print(f"  {src:<28} {total:>7,} {tr:>7,} {va:>7,} {te:>7,}")
print("-" * 64)
print(f"  {'TOTAL':<28} {len(textline_entries):>7,} {len(train_line):>7,} {len(val_line):>7,} {len(test_line):>7,}")

# Assert no intersection between splits
train_paths = set(img for img, _ in train_line)
val_paths = set(img for img, _ in val_line)
test_paths = set(img for img, _ in test_line)

assert len(train_paths & val_paths) == 0, "Leakage: train/val overlap!"
assert len(train_paths & test_paths) == 0, "Leakage: train/test overlap!"
assert len(val_paths & test_paths) == 0, "Leakage: val/test overlap!"
assert len(train_paths | val_paths | test_paths) == len(textline_entries), \
    "Entry count mismatch after split!"

print(f"\nKhông có data leakage: các giao cắt train/val/test đều rỗng.")
print(f"Tổng số mẫu được bảo toàn: {len(train_paths | val_paths | test_paths):,}")

Phân phối nguồn dữ liệu (Text Line):
  UIT_HWDB_line_flat: 7,229
  VNOnDB_line: 7,296
  vn_handwritten_images: 1,838
  Total: 16,363

Kết quả split:
  Train: 13,090  (80.0%)
  Val:   1,636  (10.0%)
  Test:  1,637  (10.0%)

Source                           Total   Train     Val    Test
----------------------------------------------------------------
  UIT_HWDB_line_flat             7,229   5,783     723     723
  VNOnDB_line                    7,296   5,837     729     730
  vn_handwritten_images          1,838   1,470     184     184
----------------------------------------------------------------
  TOTAL                         16,363  13,090   1,636   1,637

Không có data leakage: các giao cắt train/val/test đều rỗng.
Tổng số mẫu được bảo toàn: 16,363


## 5. Lưu dữ liệu đầu ra

In [12]:
# --------------------------------------------------
# 5. Lưu Outputs
# --------------------------------------------------

# Create output directory
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(f"Output directory: {os.path.abspath(CONFIG['output_dir'])}")


def save_entries(entries: list, filepath: str):
    """Save entries as tab-separated file (image_path\ttranscription)."""
    with open(filepath, "w", encoding="utf-8") as f:
        for img_path, transcription in entries:
            f.write(f"{img_path}\t{transcription}\n")
    print(f"  Saved {len(entries):,} entries to {filepath}")


# Lưu split files
print("\nĐang lưu các file split...")
save_entries(train_line, CONFIG["train_line_path"])
save_entries(val_line, CONFIG["val_line_path"])
save_entries(test_line, CONFIG["test_line_path"])

# Ràng buộc từ EDA để downstream notebooks đồng bộ
eda_constraints = {
    "image_height": 64,
    "resized_width_percentiles": {
        "p50": 980,
        "p75": 1160,
        "p90": 1250,
        "p95": 1370,
        "p99": 1580,
    },
    "recommended_image_max_width": 1370,
    "current_max_width_512_clamp_rate_pct": 93.1,
    "aspect_ratio_stats_sampled": {
        "min": 0.80,
        "max": 33.68,
        "mean": 14.97,
        "std": 4.29,
    },
}

# Lưu metadata
metadata = {
    "timestamp": datetime.now().isoformat(),
    "random_seed": CONFIG["random_seed"],
    "split_ratios": {
        "train": CONFIG["train_ratio"],
        "val": CONFIG["val_ratio"],
        "test": CONFIG["test_ratio"],
    },
    "min_image_dim": CONFIG["min_image_dim"],
    "aspect_ratio_range": [CONFIG["aspect_ratio_min"], CONFIG["aspect_ratio_max"]],
    "min_label_len": CONFIG["min_label_len"],
    "word_dataset_used": False,
    "counts": {
        "textline_total_parsed": before_count,
        "textline_invalid_images": len(textline_invalid),
        "train_line": len(train_line),
        "val_line": len(val_line),
        "test_line": len(test_line),
    },
    "source_distribution": {
        "total": dict(source_counts),
        "train": dict(train_sources),
        "val": dict(val_sources),
        "test": dict(test_sources),
    },
    "nfc_normalization": {
        "textline_changed": tl_changed,
    },
    "label_cleaning": {
        "whitespace_normalized": ws_changed,
        "control_chars_removed": ctrl_removed,
        "dropped_too_short": too_short,
    },
    "eda_constraints": eda_constraints,
}

with open(CONFIG["metadata_path"], "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"  Saved metadata to {CONFIG['metadata_path']}")

print("\nĐã lưu toàn bộ output thành công.")

Output directory: /home/khang/Projects/OCR_project/ocr_ai_agent_coding/data/processed

Đang lưu các file split...
  Saved 13,090 entries to ../data/processed/train_line.txt
  Saved 1,636 entries to ../data/processed/val_line.txt
  Saved 1,637 entries to ../data/processed/test_line.txt
  Saved metadata to ../data/processed/split_metadata.json

Đã lưu toàn bộ output thành công.


## 6. Kiểm tra lại kết quả

In [ ]:
# --------------------------------------------------
# 6. Verification
# --------------------------------------------------

print("Đang đọc lại file đã lưu để verification...\n")

# Re-read files
verify_train_line = parse_label_file(CONFIG["train_line_path"])
verify_val_line = parse_label_file(CONFIG["val_line_path"])
verify_test_line = parse_label_file(CONFIG["test_line_path"])

with open(CONFIG["metadata_path"], "r", encoding="utf-8") as f:
    verify_metadata = json.load(f)

# Verify counts
print("Kiểm tra số lượng:")
assert len(verify_train_line) == len(train_line), \
    f"train_line mismatch: {len(verify_train_line)} vs {len(train_line)}"
print(f"  train_line.txt:  {len(verify_train_line):,} (expected {len(train_line):,}) OK")

assert len(verify_val_line) == len(val_line), \
    f"val_line mismatch: {len(verify_val_line)} vs {len(val_line)}"
print(f"  val_line.txt:    {len(verify_val_line):,} (expected {len(val_line):,}) OK")

assert len(verify_test_line) == len(test_line), \
    f"test_line mismatch: {len(verify_test_line)} vs {len(test_line)}"
print(f"  test_line.txt:   {len(verify_test_line):,} (expected {len(test_line):,}) OK")

# Verify NFC normalization on re-read data
print("\nKiểm tra NFC normalization:")
for name, entries in [("train_line", verify_train_line),
                       ("val_line", verify_val_line),
                       ("test_line", verify_test_line)]:
    for img, text in entries:
        assert text == unicodedata.normalize("NFC", text), \
            f"NFC fail in {name}: {img}"
    print(f"  {name}: all {len(entries):,} transcriptions are NFC-normalized. OK")

# Verify label cleaning: no control chars, no multi-space, no leading/trailing space
print("\nKiểm tra label cleaning:")
for name, entries in [("train_line", verify_train_line),
                       ("val_line", verify_val_line),
                       ("test_line", verify_test_line)]:
    for img, text in entries:
        # No control/format characters
        for ch in text:
            assert unicodedata.category(ch) not in ("Cc", "Cf"), \
                f"Control char in {name}: {img} (U+{ord(ch):04X})"
        # No leading/trailing whitespace
        assert text == text.strip(), \
            f"Leading/trailing whitespace in {name}: {img}"
        # No multi-space runs
        assert "  " not in text, \
            f"Multi-space in {name}: {img}"
        # Length guard
        assert len(text) >= CONFIG["min_label_len"], \
            f"Label too short in {name}: {img} (len={len(text)})"
    print(f"  {name}: all {len(entries):,} labels clean. OK")

# Verify no data leakage
print("\nKiểm tra data leakage:")
verify_train_paths = set(img for img, _ in verify_train_line)
verify_val_paths = set(img for img, _ in verify_val_line)
verify_test_paths = set(img for img, _ in verify_test_line)

assert len(verify_train_paths & verify_val_paths) == 0, "Leakage: train/val overlap!"
assert len(verify_train_paths & verify_test_paths) == 0, "Leakage: train/test overlap!"
assert len(verify_val_paths & verify_test_paths) == 0, "Leakage: val/test overlap!"
print(f"  Train/Val/Test intersections: 0 OK")

# Verify Word Dataset is NOT used
assert verify_metadata.get("word_dataset_used") == False, \
    "Metadata should indicate word_dataset_used=False!"
print(f"  word_dataset_used=False in metadata: OK")

# Verify metadata counts match
print("\nKiểm tra metadata:")
mc = verify_metadata["counts"]
assert mc["train_line"] == len(train_line)
assert mc["val_line"] == len(val_line)
assert mc["test_line"] == len(test_line)
print(f"  All metadata counts match. OK")

# Verify label_cleaning stats in metadata
lc = verify_metadata.get("label_cleaning", {})
assert "whitespace_normalized" in lc, "Missing label_cleaning in metadata!"
assert "control_chars_removed" in lc, "Missing label_cleaning in metadata!"
assert "dropped_too_short" in lc, "Missing label_cleaning in metadata!"
print(f"  label_cleaning stats in metadata: OK")

# Final summary table
print("\n" + "=" * 60)
print("TỔNG KẾT CUỐI")
print("=" * 60)
print(f"{'Dataset':<25} {'Split':<12} {'Count':>10}")
print("-" * 50)
print(f"{'Text Line':<25} {'train':<12} {len(train_line):>10,}")
print(f"{'Text Line':<25} {'val':<12} {len(val_line):>10,}")
print(f"{'Text Line':<25} {'test':<12} {len(test_line):>10,}")
print("-" * 50)
print(f"{'Total':<25} {'':<12} {len(train_line) + len(val_line) + len(test_line):>10,}")
print("=" * 60)
print(f"\nWord Dataset: KHÔNG SỬ DỤNG (quyết định Phase 1)")
print("\nTất cả bước verification đều đạt.")

Đang đọc lại file đã lưu để verification...

Kiểm tra số lượng:
  train_line.txt:  14,726 (expected 14,726) OK
  val_line.txt:    1,637 (expected 1,637) OK

Kiểm tra NFC normalization:
  train_line: all 14,726 transcriptions are NFC-normalized. OK
  val_line: all 1,637 transcriptions are NFC-normalized. OK

Kiểm tra label cleaning:
  train_line: all 14,726 labels clean. OK
  val_line: all 1,637 labels clean. OK

Kiểm tra data leakage:
  Train/Val intersection: 0 OK
  word_dataset_used=False in metadata: OK

Kiểm tra metadata:
  All metadata counts match. OK
  label_cleaning stats in metadata: OK

TỔNG KẾT CUỐI
Dataset                   Split             Count
--------------------------------------------------
Text Line                 train            14,726
Text Line                 val               1,637
--------------------------------------------------
Total                                      16,363

Word Dataset: KHÔNG SỬ DỤNG (quyết định Phase 1)

Tất cả bước verification đều 

## Tổng kết


## Tổng hợp kiểm tra tuân thủ Phase 1

**Đặc tả:** `docs/phase1_notes.md`

### Hạng mục đã kiểm tra (notebook 02)

| Yêu cầu | Tham chiếu đặc tả | Trạng thái | Vị trí |
|---|---|---|---|
| Chuẩn hóa khoảng trắng | A.4 / Pipeline bước 3 | **Có** | `clean_transcription()` cell-3 → áp dụng cell-9 |
| Loại ký tự control (`Cc`, `Cf`) | A.4 / C Rule 3 | **Có** | `clean_transcription()` cell-3 → áp dụng cell-9 |
| Guard `len(label) < 2` | A.2 / C Rule 1 | **Có** | `CONFIG["min_label_len"] = 2` cell-1 → enforce cell-9 |
| Lọc aspect ratio (`<0.5`, `>50`) | C Rule 4 | **Có** | `CONFIG["aspect_ratio_min/max"]` cell-1 → `validate_images()` cell-3 → áp dụng cell-11 |

### Hạng mục mới bổ sung

Không có — cả 4 yêu cầu đã được triển khai đúng từ trước.

### Chi tiết triển khai

- **Chuẩn hóa khoảng trắng:** `re.sub(r"\\s+", " ", text).strip()` gộp multi-space và trim hai đầu.
- **Loại control char:** Lọc bằng `unicodedata.category(ch) not in ("Cc", "Cf")`.
- **Guard độ dài:** `len(cleaned) < CONFIG["min_label_len"]` thì loại mẫu (ngưỡng = 2).
- **Aspect ratio:** `validate_images()` tính `W/H` và loại ngoài khoảng `[0.5, 50.0]`.
- **Metadata:** `split_metadata.json` có `label_cleaning` và `aspect_ratio_range`.
- **Verification:** cell-18 assert không còn control chars, không multi-space, không dư khoảng trắng đầu/cuối, và `len >= 2` trên toàn bộ dữ liệu đã lưu.

### Xác nhận chạy pipeline

- Mọi key trong CONFIG đều được định nghĩa và tham chiếu đúng.
- Luồng pipeline: parse → NFC → clean (whitespace + control + length guard) → validate ảnh (integrity + aspect ratio) → split → save → verify.
- Không chỉnh sửa logic ngoài phạm vi yêu cầu.

## Mô tả từng cell (Notebook 02) và tổng kết

### Mô tả từng cell
- **Cell 1 (Markdown):** Giới thiệu mục tiêu chuẩn bị dữ liệu cho Phase 1.
- **Cell 2 (Markdown):** Ghi chú điều chỉnh split để đồng bộ với VietOCR Trainer.
- **Cell 3 (Markdown):** Ghi chú điều chỉnh về quản lý vocabulary bởi VietOCR.
- **Cell 4 (Mã):** Khai báo CONFIG (đường dẫn, tỷ lệ split 90/10, seed, ngưỡng kiểm tra, aspect ratio, min label len).
- **Cell 5 (Mã):** Import thư viện cần thiết (bao gồm `re` cho chuẩn hóa khoảng trắng).
- **Cell 6 (Mã):** Định nghĩa helper functions (parse nhãn, NFC normalize, clean_transcription, validate ảnh với aspect ratio, tách source).
- **Cell 7 (Markdown):** Mục 1 — Parse file nhãn.
- **Cell 8 (Mã):** Đọc file nhãn Text Line, in số lượng và xem mẫu dữ liệu.
- **Cell 9 (Markdown):** Mục 2 — Chuẩn hóa Unicode NFC.
- **Cell 10 (Mã):** Chuẩn hóa Unicode NFC cho transcription và kiểm tra thay đổi.
- **Cell 11 (Markdown):** Mục 2b — Làm sạch nhãn (whitespace, control chars, length guard).
- **Cell 12 (Mã):** Áp dụng clean_transcription: loại control chars, gộp multi-space, strip, loại nhãn ngắn < 2.
- **Cell 13 (Markdown):** Mục 3 — Kiểm tra tính hợp lệ ảnh.
- **Cell 14 (Mã):** Kiểm tra ảnh hợp lệ (tồn tại, mở được, kích thước tối thiểu, aspect ratio).
- **Cell 15 (Markdown):** Mục 4 — Chia tập train/validation theo stratified.
- **Cell 16 (Mã):** Chia dữ liệu 90/10 theo source, kiểm tra data leakage.
- **Cell 17 (Markdown):** Mục 5 — Lưu dữ liệu đầu ra.
- **Cell 18 (Mã):** Lưu train/val và metadata (bao gồm thống kê làm sạch nhãn + ràng buộc EDA).
- **Cell 19 (Markdown):** Mục 6 — Kiểm tra lại kết quả.
- **Cell 20 (Mã):** Đọc lại file đã lưu để xác nhận tính nhất quán (NFC, label cleaning, leakage, metadata).
- **Cell 21 (Markdown):** Tổng kết notebook.

### Tổng kết file 02
Notebook 02 hoàn tất pipeline chuẩn bị dữ liệu cho Phase 1: parse nhãn, chuẩn hóa Unicode NFC, làm sạch nhãn (chuẩn hóa khoảng trắng, loại control chars, length guard), kiểm tra ảnh (integrity + aspect ratio), chia tập dữ liệu theo stratified split (90% train / 10% validation) và xuất artifact sẵn sàng cho huấn luyện Baseline. Split chỉ gồm train + validation để tương thích với VietOCR Trainer. Vocabulary do VietOCR tự quản lý qua `Cfg.load_config_from_name()`.